In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import librosa
import parselmouth

In [2]:
nonAfricanCsv = pd.read_csv('./data/nonAfrica.csv', header=0, index_col=0)

In [3]:
nonAfricanAudio = {file:librosa.load('./data/audios_development/'+file)[0] for file in nonAfricanCsv['path'].apply(lambda x: x.split('/')[-1])}

In [4]:
def computeZcr(audio):
    zcr = librosa.feature.zero_crossing_rate(audio)
    return pd.Series([zcr.mean(), zcr.std(), zcr.max(), zcr.min()], index=['zcr_mean', 'zcr_std', 'zcr_max', 'zcr_min'])

In [5]:
def computeRMS(audio):
    rms = librosa.feature.rms(y=audio)
    return pd.Series([rms.mean(), rms.std(), rms.max(), rms.min()], index=['rms_mean', 'rms_std', 'rms_max', 'rms_min'])

In [6]:
def computeMFCC(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=13)
    return pd.Series(np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1), mfcc.max(axis=1), mfcc.min(axis=1)], axis=0), 
                                    index=np.array([[f'mfcc_mean{i}', f'mfcc_std{i}', f'mfcc_max{i}', f'mfcc_min{i}'] for i in range(0,13)]).T.flatten())

In [7]:
def computeF0(audio):
    f0 = librosa.yin(audio, fmin=100, fmax=2000)
    return pd.Series([f0.mean(), f0.std(), f0.max(), f0.min()], index=['f0_mean', 'f0_std', 'f0_max', 'f0_min'])

In [8]:
def computeSpectralFeature(audio):
    spectralCentroid = librosa.feature.spectral_centroid(y=audio)
    spectralBandwidth = librosa.feature.spectral_bandwidth(y=audio)
    spectralContrast = librosa.feature.spectral_contrast(y=audio)
    spectralRolloff = librosa.feature.spectral_rolloff(y=audio)
    return pd.Series([spectralCentroid.mean(), spectralCentroid.std(), spectralBandwidth.mean(), spectralBandwidth.std(), 
                      spectralContrast.mean(), spectralContrast.std(), spectralRolloff.mean(), spectralRolloff.std()],
                     index=['spectralCentroid_mean', 'spectralCentroid_std', 'spectralBandwidth_mean', 'spectralBandwidth_std',
                            'spectralContrast_mean', 'spectralContrast_std', 'spectralRolloff_mean', 'spectralRolloff_std'])

In [9]:
def computeSpeechRate(audio):
    intervals = librosa.effects.split(audio, top_db=20)    
    return pd.Series([np.sum(np.diff(intervals, axis=1))/audio.shape[0]], index=['speech_rate'])

In [10]:
def computePauses(audio):
    intervals = librosa.effects.split(audio, top_db=20)
    # Calculate the duration of pauses
    pause_durations = np.diff(intervals, axis=1) / 22050
    return pd.Series([intervals.shape[0], pause_durations.mean(), pause_durations.std()], 
                     index=['#pauses', 'mean_pause_duration', 'std_pause_duration'])

In [11]:
def computeJitterShimmerHNR(audio):
    jitter, shimmer = librosa.effects.jitter(audio), librosa.effects.shimmer(audio)
    HNR = librosa.effects.harmonic(audio)
    return pd.Series([jitter.mean(), jitter.std(), shimmer.mean(), shimmer.std(), HNR.mean(), HNR.std()], 
                     index=['jitter_mean', 'jitter_std', 'shimmer_mean', 'shimmer_std', 'HNR_mean', 'HNR_std'])

In [12]:
def computeFormantFeatures(audio, sr=22050):
    sound = parselmouth.Sound(audio, sr)
    formant = sound.to_formant_burg()
    
    f1 = formant.get_value_at_time(1, 0.5)
    f2 = formant.get_value_at_time(2, 0.5)
    f3 = formant.get_value_at_time(3, 0.5)
    
    b1 = formant.get_bandwidth_at_time(1, 0.5)
    b2 = formant.get_bandwidth_at_time(2, 0.5)
    b3 = formant.get_bandwidth_at_time(3, 0.5)
    
    total_energy = np.sum(audio ** 2)
    formant_energy = np.sum([f1, f2, f3])
    formant_energy_ratio = formant_energy / total_energy
    
    return pd.Series([f1, f2, f3, b1, b2, b3, formant_energy_ratio], 
                     index=['f1', 'f2', 'f3', 'b1', 'b2', 'b3', 'formant_energy_ratio'])

In [13]:
def computeChroma(audio):
    chroma = librosa.feature.chroma_stft(y=audio, sr=22050)
    return pd.Series(np.concatenate((chroma.mean(axis=1), chroma.std(axis=1)), axis=0),
                     index=np.array([[f'chroma_mean{i}', f'chroma_std{i}'] for i in range(0,12)]).T.flatten())

In [14]:
def computeTempo(audio):
    return pd.Series(librosa.feature.tempo(y=audio)[0], index=['tempo'])

In [15]:
def computeTonnetz(audio):
    tonnetto = librosa.feature.tonnetz(y=audio)
    return pd.Series(np.concatenate((tonnetto.mean(axis=1), tonnetto.std(axis=1)), axis=0),
                     index=np.array([[f'tonnetz_mean{i}', f'tonnetz_std{i}'] for i in range(0,6)]).T.flatten())

In [16]:
def computeMel(audio):
    mel = librosa.feature.melspectrogram(y=audio)
    return pd.Series(np.concatenate((mel.mean(axis=1), mel.std(axis=1)), axis=0),
                        index=np.array([[f'mel_mean{i}', f'mel_std{i}'] for i in range(0,128)]).T.flatten())

In [17]:
data = pd.DataFrame({file:pd.concat([
    computeZcr(nonAfricanAudio[file]),
    computeRMS(nonAfricanAudio[file]),
    computeMFCC(nonAfricanAudio[file]),
    computeF0(nonAfricanAudio[file]),
    computeSpectralFeature(nonAfricanAudio[file]),
    computeSpeechRate(nonAfricanAudio[file]),
    computePauses(nonAfricanAudio[file]),
    # computeJitterShimmerHNR(nonAfricanAudio[file]),
    computeFormantFeatures(nonAfricanAudio[file]),
    computeChroma(nonAfricanAudio[file]),
    computeTempo(nonAfricanAudio[file]),
    computeTonnetz(nonAfricanAudio[file]),
    computeMel(nonAfricanAudio[file])
    ], axis=0)
    for file in nonAfricanAudio}).T

In [18]:
data = pd.concat([data, nonAfricanCsv.set_index(nonAfricanCsv['path'])['age']], axis=1)

In [19]:
# data = pd.read_csv('./data/nonAfricanFeatures.csv', header=0, index_col=0)

In [20]:
corrrr = data.corr('spearman')['age']

for index in corrrr.map(np.abs).sort_values(ascending=False).index:
    print(index, corrrr[index])    

age 1.0
mel_std126 -0.25428348999767864
mel_std125 -0.2520147920041821
mel_mean126 -0.251312529604878
mel_std127 -0.24979532639366184
#pauses 0.2484159248010522
mel_mean125 -0.2476288290569373
mel_mean127 -0.24628000848649068
mel_mean118 -0.24140459822211685
mel_std124 -0.2404358817326488
mel_mean119 -0.24032420695539616
mel_std122 -0.2396943900138066
mel_std119 -0.23891835178510681
mel_mean121 -0.23820112979395075
mel_std121 -0.2378891372907427
mel_std123 -0.2378002613112265
mel_mean120 -0.23746323710656128
mel_mean122 -0.23709651811823462
mel_mean124 -0.23650555444116675
mel_std118 -0.23563986839544152
mel_mean117 -0.2355278286598778
mel_std120 -0.23540263686247567
mel_mean123 -0.2352641092911123
mel_std117 -0.2290714855721846
mel_mean116 -0.2181941398404822
mel_std116 -0.21042061165648226
speech_rate -0.20732918148565674
spectralContrast_std 0.20072218113988202
mel_mean115 -0.19893906757706664
mel_std0 -0.1959229707185511
spectralCentroid_std -0.1954352446237269
mel_std115 -0.190876

In [ ]:
data

,zcr_mean,zcr_std,zcr_max,zcr_min,rms_mean,rms_std,rms_max,rms_min,mfcc_mean0,mfcc_mean1,...,mel_std119,mel_std120,mel_std121,mel_std122,mel_std123,mel_std124,mel_std125,mel_std126,mel_std127,age
1.wav,0.210093,0.174175,0.786621,0.019043,0.031842,0.035200,0.200347,0.000241,-369.445770,55.719803,...,3.800879e-02,2.803914e-02,2.274095e-02,2.691475e-02,2.016690e-02,1.525524e-02,9.101027e-03,3.658939e-03,3.165333e-04,24.0
2.wav,0.078849,0.074397,0.397461,0.008789,0.069174,0.055064,0.238681,0.003501,-291.229675,122.431274,...,1.008640e-03,8.167035e-04,8.310446e-04,1.040648e-03,1.315697e-03,1.215531e-03,1.279807e-03,8.888561e-04,5.950056e-05,22.5
3.wav,0.105365,0.120708,0.574219,0.003418,0.044096,0.032049,0.186966,0.004962,-304.470520,72.080658,...,2.357865e-02,1.842017e-02,1.756822e-02,1.664084e-02,9.081427e-03,9.565935e-03,9.624348e-03,6.201901e-03,6.023237e-04,22.0
4.wav,0.173701,0.167091,0.799805,0.000000,0.121360,0.087285,0.379289,0.000018,-254.585403,38.093533,...,5.876975e-01,1.143306e+00,1.709680e+00,1.085319e+00,7.186143e-01,5.942925e-01,6.352082e-01,1.906777e-01,1.844911e-02,22.0
5.wav,0.107279,0.105950,0.498047,0.005371,0.059892,0.042143,0.183311,0.001987,-273.523865,112.785225,...,6.021491e-03,4.164151e-03,4.537857e-03,4.062913e-03,3.183328e-03,2.668071e-03,2.939539e-03,1.538673e-03,1.633528e-04,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,0.077740,0.103368,0.635742,0.006836,0.077101,0.057661,0.358437,0.011932,-276.933838,135.242386,...,8.016796e-02,7.473018e-02,5.930423e-02,7.320457e-02,8.508126e-02,2.014178e-01,1.338326e-01,5.994967e-02,3.632866e-03,27.0
2926.wav,0.135405,0.113255,0.576172,0.020020,0.041325,0.040323,0.240033,0.000111,-332.338348,98.117645,...,2.269334e-10,1.972600e-10,1.760291e-10,1.601321e-10,1.481141e-10,1.391611e-10,1.325423e-10,1.280395e-10,1.253108e-10,22.0
2928.wav,0.111901,0.116163,0.545898,0.018555,0.074255,0.050410,0.212878,0.009050,-227.502090,128.541199,...,1.128745e-06,9.457501e-07,8.026810e-07,6.774349e-07,5.949678e-07,5.345842e-07,4.376668e-07,3.377193e-07,9.540887e-08,39.0
2929.wav,0.111799,0.103551,0.485352,0.015625,0.035550,0.024755,0.132884,0.000972,-303.633423,100.347366,...,5.534558e-03,5.022744e-03,4.600589e-03,3.465061e-03,1.789898e-03,9.638647e-04,7.318112e-04,3.234728e-04,2.300417e-05,24.0


In [22]:
for col, impor in sorted(zip(data.drop(columns='age').columns, RandomForestRegressor(n_jobs=-1).fit(data.drop(columns=['age']), data['age']).feature_importances_), key=lambda x: x[1], reverse=True):
    print(col, impor)

#pauses 0.033291510847527565
mel_mean120 0.01031548831471162
mfcc_max4 0.008594678722657411
mel_std0 0.00854069813127445
mfcc_min6 0.008499285789015576
mel_mean5 0.00846672644185032
mel_std121 0.008241359381554472
chroma_mean5 0.008219656004701035
mel_std5 0.008105401993691287
mfcc_max9 0.007764648835436157
b1 0.007555981426813023
spectralContrast_std 0.007307006147323889
mfcc_mean7 0.007225309094851698
chroma_mean4 0.007201598085615537
mel_mean118 0.006186216292772341
mfcc_std6 0.006097819815572396
mfcc_std5 0.00607917201233716
chroma_mean3 0.006006382610319715
mel_mean121 0.005960223322191126
mel_mean12 0.005815787895757613
mel_std13 0.005780861241803097
mel_mean8 0.005717241804324016
chroma_std4 0.005708619584976865
mel_mean0 0.005665318786000502
mfcc_std9 0.005657960990728189
mfcc_mean6 0.005497970506686972
mel_mean126 0.005484370766743126
mfcc_max2 0.005480369485119724
mfcc_min9 0.005331850707959049
mfcc_min11 0.005301968704024249
chroma_mean6 0.00529294410461991
tonnetz_mean3 0.0

In [116]:
cross_val_score(RandomForestRegressor(n_jobs=-1), data.drop(columns=['age']), data['age'], cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-10.658920353982301)

In [24]:
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor

cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(n_jobs=-1)), 
                data[[col for col in data.columns if 'mel' in col]], data['age'], cv=36, scoring='neg_mean_absolute_error').mean()


np.float64(-11.704707200551617)

In [265]:
cross_val_score(make_pipeline(StandardScaler(), RandomForestRegressor(n_jobs=-1)), 
                tempdata[tempdata['gender']==0][data.columns].drop(columns=['age']), 
                tempdata[tempdata['gender']==0]['age'], 
                cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-10.531425287356322)

In [38]:
beat = {file:np.concatenate(librosa.beat.beat_track(y=nonAfricanAudio[file], sr=22050, trim=True), axis=0)
                     for file in nonAfricanAudio}

In [79]:
minBeat =np.min([len(beat[file]) for file in beat])
normalizedBeat = pd.DataFrame({file:np.concatenate(([beat[file][0]], (  
                beat[file][1:]
                if beat[file].shape[0] == minBeat
                else [np.mean(beat[file][1+beat[file].shape[0]*i//minBeat:1+min(beat[file].shape[0], beat[file].shape[0]*(i+1)//minBeat)]) for i in range(0, minBeat-1)]
            )), axis=0)  for file in beat},
            index=[f'normBeat{i}' for i in range(0, minBeat)]).T

In [64]:
# Apply the chroma filterbank to the audio signal
chroma_filter = librosa.filters.chroma(sr=22050, n_fft=2048, n_chroma=12, octwidth=None)
filtered_audio = pd.DataFrame({file:np.dot(chroma_filter, nonAfricanAudio[file][:1025]) for file in nonAfricanAudio},
                               index=[f'chroma_filter{i}' for i in range(0,12)] ).T


In [84]:
data

,zcr_mean,zcr_std,zcr_max,zcr_min,rms_mean,rms_std,rms_max,rms_min,mfcc_mean0,mfcc_mean1,...,mel_std119,mel_std120,mel_std121,mel_std122,mel_std123,mel_std124,mel_std125,mel_std126,mel_std127,age
1.wav,0.210093,0.174175,0.786621,0.019043,0.031842,0.035200,0.200347,0.000241,-369.445770,55.719803,...,3.800879e-02,2.803914e-02,2.274095e-02,2.691475e-02,2.016690e-02,1.525524e-02,9.101027e-03,3.658939e-03,3.165333e-04,24.0
2.wav,0.078849,0.074397,0.397461,0.008789,0.069174,0.055064,0.238681,0.003501,-291.229675,122.431274,...,1.008640e-03,8.167035e-04,8.310446e-04,1.040648e-03,1.315697e-03,1.215531e-03,1.279807e-03,8.888561e-04,5.950056e-05,22.5
3.wav,0.105365,0.120708,0.574219,0.003418,0.044096,0.032049,0.186966,0.004962,-304.470520,72.080658,...,2.357865e-02,1.842017e-02,1.756822e-02,1.664084e-02,9.081427e-03,9.565935e-03,9.624348e-03,6.201901e-03,6.023237e-04,22.0
4.wav,0.173701,0.167091,0.799805,0.000000,0.121360,0.087285,0.379289,0.000018,-254.585403,38.093533,...,5.876975e-01,1.143306e+00,1.709680e+00,1.085319e+00,7.186143e-01,5.942925e-01,6.352082e-01,1.906777e-01,1.844911e-02,22.0
5.wav,0.107279,0.105950,0.498047,0.005371,0.059892,0.042143,0.183311,0.001987,-273.523865,112.785225,...,6.021491e-03,4.164151e-03,4.537857e-03,4.062913e-03,3.183328e-03,2.668071e-03,2.939539e-03,1.538673e-03,1.633528e-04,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2924.wav,0.077740,0.103368,0.635742,0.006836,0.077101,0.057661,0.358437,0.011932,-276.933838,135.242386,...,8.016796e-02,7.473018e-02,5.930423e-02,7.320457e-02,8.508126e-02,2.014178e-01,1.338326e-01,5.994967e-02,3.632866e-03,27.0
2926.wav,0.135405,0.113255,0.576172,0.020020,0.041325,0.040323,0.240033,0.000111,-332.338348,98.117645,...,2.269334e-10,1.972600e-10,1.760291e-10,1.601321e-10,1.481141e-10,1.391611e-10,1.325423e-10,1.280395e-10,1.253108e-10,22.0
2928.wav,0.111901,0.116163,0.545898,0.018555,0.074255,0.050410,0.212878,0.009050,-227.502090,128.541199,...,1.128745e-06,9.457501e-07,8.026810e-07,6.774349e-07,5.949678e-07,5.345842e-07,4.376668e-07,3.377193e-07,9.540887e-08,39.0
2929.wav,0.111799,0.103551,0.485352,0.015625,0.035550,0.024755,0.132884,0.000972,-303.633423,100.347366,...,5.534558e-03,5.022744e-03,4.600589e-03,3.465061e-03,1.789898e-03,9.638647e-04,7.318112e-04,3.234728e-04,2.300417e-05,24.0


In [118]:
from sklearn.ensemble import HistGradientBoostingRegressor


cross_val_score(HistGradientBoostingRegressor(), pd.concat([
                                                            normalizedBeat, 
                                                            data.drop(columns=['age']),
                                                            
                                                            ], axis=1), data['age'], cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-9.803619501172014)

In [126]:
dev = pd.read_csv('./data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
dev['gender'] = dev['gender'].map(lambda x: 0 if x=='male' else 1)
dev['tempo'] = dev['tempo'].map(lambda x: float(x[1:-1]))

predict = pd.read_csv('./data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
predict['gender'] = predict['gender'].map(lambda x: 0 if x=='male' else 1)
predict['tempo'] = predict['tempo'].map(lambda x: float(x[1:-1]))

In [135]:
cross_val_score(HistGradientBoostingRegressor(categorical_features=dev.drop(columns=['age', 'path']).dtypes=='object'), 
                dev.drop(columns=['age', 'path'])[dev['num_words']<60], 
                dev[dev['num_words']<40]['age'], cv=10, scoring='neg_mean_absolute_error').mean()

np.float64(-3.228220422809062)

In [138]:
predictAfrican = pd.Series(HistGradientBoostingRegressor(categorical_features=dev.drop(columns=['age', 'path']).dtypes=='object')
                  .fit(dev.drop(columns=['age', 'path'])[dev['num_words']<60], dev[dev['num_words']<60]['age'])
                  .predict(predict.drop(columns=['path'])[predict['num_words']<60]),
                  name='Predicted',
                  index=predict[predict['num_words']<60].index)

In [ ]:
predictNonAfrican = pd.read_csv('./data/predicted.csv', header=0, index_col=0)
predictNonAfrican = pd.Series(predictNonAfrican.values[:, 0], index=predictNonAfrican.index, name='Predicted')